# IRS Data Analysis
#### About:
- This notebook loads the IRS datasets, processes the amount variables, and performs a descriptive analysis.
- All steps follow the instructions set in data/Tutorial_regressions_2025 but with a python implementation.

In [ ]:
# Data manipulation libraries
import pandas as pd
import numpy as np
import os

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Automated EDA library
from ydata_profiling import ProfileReport

## Load Data

In [ ]:
# Load the dataset

file_path = 'data/IRS2020.csv'

try:
    df = pd.read_csv(file_path, sep=';')
    print("Data loaded successfully.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")

## (a) Data Preprocessing

Convert all amount variables (those not starting with 'Num' and not identifiers) into dollar amounts by multiplying by 1000.

In [ ]:
# Identify columns to convert
# Identifiers: STATEFIPS, STATE, ZIPCODE
# Count variables start with 'Num'

identifiers = ['STATEFIPS', 'STATE', 'ZIPCODE']
amount_cols = [col for col in df.columns if not col.startswith('Num') and col not in identifiers]

print(f"Amount variables to convert: {amount_cols}")

# Convert to dollars
for col in amount_cols:
    df[col] = df[col] * 1000

df[amount_cols].head()

## Descriptive Analysis with ydata-profiling

Generate comprehensive summary report using `ydata-profiling` (formerly pandas-profiling).

In [ ]:
# Generate comprehensive EDA report
profile = ProfileReport(
    df,
    title="IRS 2020 Data - Exploratory Data Analysis",
    minimal=True,  # Use minimal mode for faster execution on large datasets
    explorative=True,
    samples={'head': True, 'tail': True}  # Include sample rows
)

# Display the interactive report
profile.to_notebook_iframe()

## Quick Statistics Summary Table

For a quick tabular overview of amount variables:

In [ ]:
# Extended descriptive statistics
stats = df[amount_cols].describe(
    include='all',
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]
).round(2)

# Add additional statistics
additional_stats = pd.DataFrame({
    'variance': df[amount_cols].var(),
    'skewness': df[amount_cols].skew(),
    'kurtosis': df[amount_cols].kurtosis()
})

# Combine statistics
full_stats = pd.concat([stats, additional_stats])
print("Descriptive Statistics (in Dollars):")
display(full_stats)

## (b) Household Averages

Create variables of household (HH) averages in thousands of $ per ZIP code (Amount/Number; NumofReturn – total number of returns; for dividends and real estate (RE) taxes only calculate averages for those HH that file they have dividends or pay RE taxes) of income, salary income, dividends, and real estate taxes paid.

In [ ]:
# Note: df[amount_cols] are already in dollars from step (a)
# Instructions ask for averages in thousands of $

df['HH_Income'] = (df['TotalIncome'] / 1000) / df['NumofReturns']
df['HH_Salaries'] = (df['Salaries'] / 1000) / df['NumofReturns']

# For dividends and RE taxes, only calculate for those HH that file they have dividends or pay RE taxes
df['HH_Dividends'] = (df['Divis'] / 1000) / df['NumofRetwithDivis']
df['HH_RE_Taxes'] = (df['Retax'] / 1000) / df['NumofRetwithRE']

hh_cols = ['HH_Income', 'HH_Salaries', 'HH_Dividends', 'HH_RE_Taxes']

print("Distributions of Household Averages (in thousands of $):")
display(df[hh_cols].describe())

# Boxplots of household averages across ZIP codes
plt.figure(figsize=(12, 6))
df[hh_cols].boxplot()
plt.title("Boxplots of Household Averages across ZIP codes")
plt.ylabel("Thousands of $")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## (c) Highest and Lowest Averages

Report the ZIP codes with the highest and the lowest average HH income, HH Salaries, HH Dividends, and HH Real estate taxes paid.

In [ ]:
for col in hh_cols:
    # Filter out potential Infs/NaNs from division by zero if any (though NumofReturns should be > 0)
    valid_df = df[df[col].notna() & (df[col] != float('inf'))]
    
    highest = valid_df.loc[valid_df[col].idxmax()]
    lowest = valid_df.loc[valid_df[col].idxmin()]
    
    print(f"--- {col} ---")
    print(f"Highest: ZIP {int(highest['ZIPCODE'])} ({highest['STATE']}) - ${highest[col]:.2f}k")
    print(f"Lowest:  ZIP {int(lowest['ZIPCODE'])} ({lowest['STATE']}) - ${lowest[col]:.2f}k\n")

## (d) Fraction of Households Receiving Dividends

Calculate the fraction of households receiving dividends (NumofRetwithdivis/NumofReturn). Report the distribution of this variable. Plot a scatterplot of HH income and fraction of HH receiving dividends. Also with colors by US State.

In [ ]:
df['Fraction_Divis'] = df['NumofRetwithDivis'] / df['NumofReturns']

print("Distribution of Fraction of HH receiving dividends:")
display(df['Fraction_Divis'].describe())

# Scatterplot of HH income and fraction of HH receiving dividends
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='HH_Income', y='Fraction_Divis', alpha=0.4)
plt.title("HH Income vs Fraction of HH Receiving Dividends")
plt.xlabel("HH Average Income (Thousands of $)")
plt.ylabel("Fraction of HH receiving dividends")
plt.show()

# Plot with colors by US State (Top 10 states for clarity)
top_states = df['STATE'].value_counts().nlargest(10).index
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df[df['STATE'].isin(top_states)], 
                x='HH_Income', y='Fraction_Divis', hue='STATE', alpha=0.5)
plt.title("HH Income vs Fraction of HH Receiving Dividends (Top 10 States)")
plt.xlabel("HH Average Income (Thousands of $)")
plt.ylabel("Fraction of HH receiving dividends")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()